# Week 4: LLM-Based Architectural Recovery — Group 5
## Hadoop YARN — Scheduler / Capacity

**Model:** `ByteDance-Seed/Seed-Coder-8B-Instruct`

### 3-Level Hierarchical Summarisation
```
Level 1 — Leaf   : each .java file        → file summary
Level 2 — Subdir : allocator/, conf/,      → subdir summary
                   placement/, policy/,
                   preemption/, queuemanagement/
Level 3 — Branch : whole cluster           → title + description (CSV)
```

### Before running
1. Runtime > Change runtime type > **T4 GPU**
2. Key icon (left sidebar) > Add secret: **`HF_TOKEN`**
3. Upload **`arc_clusters.rsf`** to `/content/` (rename your `final_clustering_jina_code_embeddings.rsf`)

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers accelerate bitsandbytes>=0.46.1
print('Done.')

Done.


In [ ]:
# Cell 2: Mount Drive + Clone Hadoop + Configure paths
import subprocess
from pathlib import Path
from google.colab import drive

# Mount Drive so checkpoints survive disconnects
drive.mount('/content/drive')
OUTPUT_DIR = Path('/content/drive/MyDrive/week4_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

# Clone Hadoop (shallow)
if not Path('/content/hadoop').exists():
    print('Cloning Hadoop (shallow, ~1-2 min)...')
    subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/apache/hadoop.git'],
        cwd='/content', check=True
    )
    print('Clone done.')
else:
    print('Hadoop already cloned.')

# Source root — keep at src/main/java so dotted package names match RSF keys
SOURCE_ROOT = Path(
    '/content/hadoop/hadoop-yarn-project/hadoop-yarn/hadoop-yarn-server/'
    'hadoop-yarn-server-resourcemanager/src/main/java'
)

# Capacity folder — used to detect subdirectory structure
CAPACITY_DIR = (
    SOURCE_ROOT
    / 'org/apache/hadoop/yarn/server/resourcemanager/scheduler/capacity'
)

# RSF files — upload to /content/ with these names
RSF_FILES = {
    'ARC':   Path('/content/arc_clusters.rsf'),
    'ACDC':  Path('/content/acdc_clusters.rsf'),
    'LIMBO': Path('/content/limbo_clusters.rsf'),
}

LIGHTWEIGHT_MODEL = 'ByteDance-Seed/Seed-Coder-8B-Instruct'

for label, p in [('SOURCE_ROOT', SOURCE_ROOT), ('CAPACITY_DIR', CAPACITY_DIR)]:
    status = 'OK' if p.exists() else 'NOT FOUND'
    print(f'  {label}: {status}')

Mounted at /content/drive
Output dir: /content/drive/MyDrive/week4_outputs
Cloning Hadoop (shallow, ~1-2 min)...
Clone done.
  SOURCE_ROOT: OK
  CAPACITY_DIR: OK


In [ ]:
# Cell 3: Load LLM in 4-bit mode
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata

if not hasattr(transformers.utils.generic, 'retry'):
    transformers.utils.generic.retry = lambda *a, **kw: lambda f: f

try:
    hf_token = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded.')
except Exception:
    hf_token = None
    print('HF_TOKEN not found.')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print(f'Loading {LIGHTWEIGHT_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(LIGHTWEIGHT_MODEL, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    LIGHTWEIGHT_MODEL,
    quantization_config=bnb_config,
    token=hf_token,
    device_map='auto',
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
print('Model loaded!')


def query_llm(prompt_text: str, max_tokens: int = 350) -> str:
    try:
        messages = [{'role': 'user', 'content': prompt_text}]
        templated = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(
            templated, return_tensors='pt', padding=True,
            truncation=True, max_length=6000,
        ).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_new_tokens=max_tokens,
                do_sample=True, temperature=0.2, top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = outputs[0][inputs['input_ids'].shape[1]:]
        return tokenizer.decode(generated, skip_special_tokens=True).strip()
    except Exception as e:
        print(f'LLM error: {e}')
        return ''

HF_TOKEN loaded.
Loading ByteDance-Seed/Seed-Coder-8B-Instruct...


config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded!


In [ ]:
# Cell 4: Helper functions — RSF parser + file mapper
import json


def parse_rsf(rsf_path: Path) -> dict:
    """
    Parse RSF files — handles all 3 formats automatically:
      ARC:   contain Cluster_0                         org.apache...ClassName
      LIMBO: contain 0                                  org.apache...ClassName
      ACDC:  contain org.apache...capacity.ss           org.apache...ClassName

    Normalises cluster IDs to 'Cluster_0', 'Cluster_1', etc.
    Strips inner-class suffix ($Inner) for file-level mapping.
    Returns {class_name: cluster_id}
    """
    assignments = {}
    raw_cluster_ids = set()

    # First pass: collect all raw cluster IDs to detect format
    with rsf_path.open('r', encoding='utf-8') as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) >= 3 and parts[0] == 'contain':
                raw_cluster_ids.add(parts[1])

    # Detect format and build normalisation map
    cluster_id_map = {}
    if any('.' in cid for cid in raw_cluster_ids):  # ACDC: package-path cluster IDs
        for i, cid in enumerate(sorted(raw_cluster_ids)):
            cluster_id_map[cid] = f'Cluster_{i}'
        fmt = 'ACDC'
    elif all(cid.isdigit() for cid in raw_cluster_ids):  # LIMBO: numeric cluster IDs
        for cid in raw_cluster_ids:
            cluster_id_map[cid] = f'Cluster_{cid}'
        fmt = 'LIMBO'
    else:  # ARC: already Cluster_X format
        for cid in raw_cluster_ids:
            cluster_id_map[cid] = cid
        fmt = 'ARC'

    print(f'  Format detected: {fmt}')
    print(f'  Clusters: {sorted(set(cluster_id_map.values()))}')

    # Second pass: build assignments with normalised cluster IDs
    with rsf_path.open('r', encoding='utf-8') as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) < 3 or parts[0] != 'contain':
                continue
            raw_cid, class_name = parts[1], parts[2]
            cluster_id = cluster_id_map.get(raw_cid, raw_cid)
            assignments[class_name] = cluster_id
            # Strip inner class suffix for file-level match
            assignments[class_name.split('$')[0]] = cluster_id

    return assignments


def build_cluster_to_files(assignments: dict, source_root: Path) -> dict:
    """
    Walk all .java files under source_root.
    Match via dotted package name or bare stem.
    Returns {cluster_id: [Path, ...]}
    """
    cluster_to_files = {}
    all_java = sorted(source_root.rglob('*.java'))
    matched = 0
    for fpath in all_java:
        rel    = fpath.relative_to(source_root)
        dotted = str(rel.with_suffix('')).replace('/', '.').replace('\\', '.')
        cid    = assignments.get(dotted) or assignments.get(fpath.stem)
        if cid:
            cluster_to_files.setdefault(cid, []).append(fpath)
            matched += 1
    print(f'  Matched {matched}/{len(all_java)} files to clusters.')
    return cluster_to_files


def get_subdir(fpath: Path, capacity_dir: Path) -> str:
    """
    Returns immediate subdirectory name under capacity_dir,
    or 'root' if file is directly in capacity_dir.
    """
    try:
        parts = fpath.relative_to(capacity_dir).parts
        return parts[0] if len(parts) > 1 else 'root'
    except ValueError:
        return 'root'


print('Cell 4 complete.')

Cell 4 complete.


In [ ]:
# Cell 5: LEVEL 1 — Leaf node: summarise each .java file
# Passes raw source code to LLM
# Saves checkpoint to Drive after every file

FILE_SUMMARY_PROMPT = """\
You are a software architecture analyst. Analyse the following Java source file and produce
a concise semantic summary with EXACTLY these four points:
- Key Functionality: what this class does
- Core Logic: the main algorithm or design pattern used
- Inputs/Outputs: what it receives and what it produces or returns
- Dependencies: key classes, interfaces, or frameworks it relies on

Keep the total summary to 4-6 sentences.

Java file: {filename}
```java
{source_code}
```
"""

MAX_SOURCE_CHARS = 8000


def summarise_file(fpath: Path) -> str:
    try:
        source = fpath.read_text(encoding='utf-8', errors='replace')[:MAX_SOURCE_CHARS]
    except Exception as e:
        return f'[Could not read: {e}]'
    return query_llm(
        FILE_SUMMARY_PROMPT.format(filename=fpath.name, source_code=source),
        max_tokens=250
    )


def run_leaf_summarisation(algo_name: str, cluster_to_files: dict) -> dict:
    save_path = OUTPUT_DIR / f'file_summaries_{algo_name}.json'
    all_summaries = json.loads(save_path.read_text()) if save_path.exists() else {}
    if save_path.exists():
        print(f'  Resuming from checkpoint: {save_path}')

    total = sum(len(v) for v in cluster_to_files.values())
    idx   = 0
    for cid, files in sorted(cluster_to_files.items()):
        cluster_sums = all_summaries.setdefault(cid, {})
        for fpath in files:
            idx += 1
            if fpath.name in cluster_sums:
                continue
            print(f'  [{idx}/{total}] {cid} -> {fpath.name}')
            cluster_sums[fpath.name] = summarise_file(fpath)
            # Checkpoint after every file
            save_path.write_text(json.dumps(all_summaries, indent=2, ensure_ascii=False))

    print(f'Level 1 complete -> {save_path}')
    return all_summaries


# --- RUN FOR ARC ---
CURRENT_ALGO = 'ARC'

if not RSF_FILES[CURRENT_ALGO].exists():
    raise FileNotFoundError('Upload arc_clusters.rsf to /content/')

print(f'Parsing {CURRENT_ALGO} RSF...')
assignments = parse_rsf(RSF_FILES[CURRENT_ALGO])
print(f'  {len(assignments)} mappings.')

print('Building cluster->files map...')
cluster_to_files = build_cluster_to_files(assignments, SOURCE_ROOT)
for cid, flist in sorted(cluster_to_files.items()):
    print(f'  {cid}: {len(flist)} files')

print('\nLevel 1: Leaf summarisation (longest step)...')
file_summaries = run_leaf_summarisation(CURRENT_ALGO, cluster_to_files)

Parsing ARC RSF...
  Format detected: ARC
  Clusters: ['Cluster_0', 'Cluster_1', 'Cluster_2', 'Cluster_3', 'Cluster_4', 'Cluster_5', 'Cluster_6', 'Cluster_7', 'Cluster_8', 'Cluster_9']
  99 mappings.
Building cluster->files map...
  Matched 98/749 files to clusters.
  Cluster_0: 4 files
  Cluster_1: 20 files
  Cluster_2: 10 files
  Cluster_3: 23 files
  Cluster_4: 12 files
  Cluster_5: 7 files
  Cluster_6: 1 files
  Cluster_7: 3 files
  Cluster_8: 1 files
  Cluster_9: 17 files

Level 1: Leaf summarisation (longest step)...
  [1/98] Cluster_0 -> ConfiguredNodeLabels.java


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  [2/98] Cluster_0 -> QueueNodeLabelsSettings.java
  [3/98] Cluster_0 -> UserInfo.java
  [4/98] Cluster_0 -> UserWeights.java
  [5/98] Cluster_1 -> AbstractAutoCreatedLeafQueue.java
  [6/98] Cluster_1 -> AbstractManagedParentQueue.java
  [7/98] Cluster_1 -> AutoCreatedLeafQueue.java
  [8/98] Cluster_1 -> AutoCreatedLeafQueueConfig.java
  [9/98] Cluster_1 -> AutoCreatedQueueManagementPolicy.java
  [10/98] Cluster_1 -> AutoCreatedQueueTemplate.java
  [11/98] Cluster_1 -> LeafQueue.java
  [12/98] Cluster_1 -> ManagedParentQueue.java
  [13/98] Cluster_1 -> ParentQueue.java
  [14/98] Cluster_1 -> PlanQueue.java
  [15/98] Cluster_1 -> QueueManagementChange.java
  [16/98] Cluster_1 -> QueuePath.java
  [17/98] Cluster_1 -> QueuePrefixes.java
  [18/98] Cluster_1 -> QueueStateHelper.java
  [19/98] Cluster_1 -> QueueUpdateWarning.java
  [20/98] Cluster_1 -> ReservationQueue.java
  [21/98] Cluster_1 -> PriorityUtilizationQueueOrderingPolicy.java
  [22/98] Cluster_1 -> QueueOrderingPolicy.java
  [2

In [ ]:
# Cell 6: LEVEL 2 — Subdirectory node summarisation
# Groups files by subdir (allocator, conf, placement, policy, preemption, queuemanagement)
# Passes file summaries (NOT raw code) to LLM for subdir-level summary

SUBDIR_SUMMARY_PROMPT = """\
You are a software architecture analyst.
Below are semantic summaries of all Java files in the subdirectory '{subdir_name}'.

Based ONLY on these summaries (do NOT pass raw code), write a concise technical
description (4-6 sentences) of what this subdirectory is responsible for as an
architectural module. Cover:
- Its overall responsibility within the Capacity Scheduler
- How the files/components inside it collaborate
- Any notable design patterns or interfaces used

--- File summaries for '{subdir_name}/' ---
{summaries_text}
"""


def run_subdir_summarisation(algo_name: str, file_summaries: dict,
                              cluster_to_files: dict) -> dict:
    save_path = OUTPUT_DIR / f'subdir_summaries_{algo_name}.json'
    all_subdir = json.loads(save_path.read_text()) if save_path.exists() else {}
    if save_path.exists():
        print(f'  Resuming from checkpoint: {save_path}')

    for cid, files in sorted(cluster_to_files.items()):
        print(f'\nSubdir processing for {cid}...')
        cluster_subdir = all_subdir.setdefault(cid, {})

        # Group files by their immediate subdirectory
        subdir_to_files = {}
        for fpath in files:
            sd = get_subdir(fpath, CAPACITY_DIR)
            if sd != 'root':  # root-level files skip subdir summarisation
                subdir_to_files.setdefault(sd, []).append(fpath)

        for sd_name, sd_files in sorted(subdir_to_files.items()):
            if sd_name in cluster_subdir:
                print(f'  {sd_name}/ already done.')
                continue

            file_sums = file_summaries.get(cid, {})
            summaries_text = '\n\n'.join(
                f'### {f.name}\n{file_sums.get(f.name, "(no summary)")}'
                for f in sd_files
            )
            if len(summaries_text) > 8000:
                summaries_text = summaries_text[:8000] + '\n[truncated]'

            print(f'  Summarising {sd_name}/ ({len(sd_files)} files)...')
            cluster_subdir[sd_name] = query_llm(
                SUBDIR_SUMMARY_PROMPT.format(
                    subdir_name=sd_name,
                    summaries_text=summaries_text
                ),
                max_tokens=200
            )

        # Checkpoint after each cluster
        save_path.write_text(json.dumps(all_subdir, indent=2, ensure_ascii=False))

    print(f'\nLevel 2 complete -> {save_path}')
    return all_subdir


print('Level 2: Subdirectory summarisation...')
subdir_summaries = run_subdir_summarisation(CURRENT_ALGO, file_summaries, cluster_to_files)

Level 2: Subdirectory summarisation...

Subdir processing for Cluster_0...

Subdir processing for Cluster_1...
  Summarising policy/ (2 files)...
  Summarising queuemanagement/ (2 files)...

Subdir processing for Cluster_2...
  Summarising allocator/ (4 files)...
  Summarising preemption/ (3 files)...

Subdir processing for Cluster_3...
  Summarising conf/ (1 files)...

Subdir processing for Cluster_4...
  Summarising conf/ (11 files)...

Subdir processing for Cluster_5...
  Summarising allocator/ (1 files)...
  Summarising conf/ (2 files)...
  Summarising placement/ (1 files)...
  Summarising queuemanagement/ (1 files)...

Subdir processing for Cluster_6...

Subdir processing for Cluster_7...
  Summarising placement/ (3 files)...

Subdir processing for Cluster_8...

Subdir processing for Cluster_9...

Level 2 complete -> /content/drive/MyDrive/week4_outputs/subdir_summaries_ARC.json


In [ ]:
# Cell 7: LEVEL 3 — Cluster branch node: final title + description
# Combines subdir summaries + root-level file summaries
# Outputs title + <=150 word description per cluster

CLUSTER_SUMMARY_PROMPT = """\
You are a software architecture expert performing architectural recovery on
Hadoop YARN Capacity Scheduler.

Below are summaries of the subdirectories and root-level files belonging to
one architectural cluster. Based ONLY on these summaries, generate:

1. TITLE: A short precise architectural title (5-10 words).
2. DESCRIPTION: A concise summary STRICTLY under 150 words that covers:
   a) Components and Interactions: how the parts work together
   b) Quality Attributes: non-functional properties (e.g. scalability, fairness,
      security, maintainability, extensibility)
   c) Technology Used: Java patterns, frameworks, or tools identified

Format your response EXACTLY like this with no extra text:
TITLE: <title here>
DESCRIPTION: <description here>

--- Summaries for Cluster {cluster_id} ---
{summaries_text}
"""


def parse_cluster_response(response: str):
    title, desc_parts, in_desc = 'N/A', [], False
    for line in response.splitlines():
        if line.startswith('TITLE:'):
            title = line[6:].strip()
        elif line.startswith('DESCRIPTION:'):
            in_desc = True
            desc_parts.append(line[12:].strip())
        elif in_desc and line.strip():
            desc_parts.append(line.strip())
    return title, ' '.join(desc_parts) if desc_parts else 'N/A'


def run_cluster_summarisation(algo_name, file_summaries, subdir_summaries, cluster_to_files):
    rows = []
    total = len(cluster_to_files)

    for i, (cid, files) in enumerate(sorted(cluster_to_files.items()), 1):
        print(f'\n[{i}/{total}] Cluster summary: {cid}')
        parts = []

        # Add subdir-level summaries (Level 2 output)
        for sd_name, sd_sum in sorted(subdir_summaries.get(cid, {}).items()):
            parts.append(f'### Subdirectory: {sd_name}/\n{sd_sum}')

        # Add root-level file summaries (files directly in capacity/, no subdir)
        file_sums = file_summaries.get(cid, {})
        for fpath in files:
            if get_subdir(fpath, CAPACITY_DIR) == 'root':
                s = file_sums.get(fpath.name, '')
                if s:
                    parts.append(f'### Root file: {fpath.name}\n{s}')

        if not parts:
            print(f'  No summaries for {cid}, skipping.')
            continue

        summaries_text = '\n\n'.join(parts)
        if len(summaries_text) > 10000:
            summaries_text = summaries_text[:10000] + '\n[truncated]'

        response = query_llm(
            CLUSTER_SUMMARY_PROMPT.format(cluster_id=cid, summaries_text=summaries_text),
            max_tokens=300
        )
        print(f'  Preview: {response[:150]}...')

        title, description = parse_cluster_response(response)
        rows.append({
            'cluster_ID':  cid,
            'files':       '; '.join(sorted(f.name for f in files)),
            'title':       title,
            'description': description,
        })

    return rows


print('Level 3: Cluster-level summarisation...')
cluster_results = run_cluster_summarisation(
    CURRENT_ALGO, file_summaries, subdir_summaries, cluster_to_files
)
print(f'Done — {len(cluster_results)} clusters.')

Level 3: Cluster-level summarisation...

[1/10] Cluster summary: Cluster_0
  Preview: TITLE: Hadoop YARN Capacity Scheduler - Node Label Management and User Scheduling

DESCRIPTION: The Hadoop YARN Capacity Scheduler architecture focuse...

[2/10] Cluster summary: Cluster_1
  Preview: TITLE: Hadoop YARN Capacity Scheduler Architecture

DESCRIPTION: The Hadoop YARN Capacity Scheduler implements a hierarchical queue system for resourc...

[3/10] Cluster summary: Cluster_2
  Preview: TITLE: Hadoop YARN Capacity Scheduler Architecture

DESCRIPTION: The Hadoop YARN Capacity Scheduler is a sophisticated resource management system that...

[4/10] Cluster summary: Cluster_3
  Preview: TITLE: Hadoop YARN Capacity Scheduler Architecture

DESCRIPTION: The Hadoop YARN Capacity Scheduler is a sophisticated resource management system that...

[5/10] Cluster summary: Cluster_4
  Preview: TITLE: Hadoop YARN Capacity Scheduler Configuration Management

DESCRIPTION: The Hadoop YARN Capacity Scheduler Co

In [ ]:
# Cell 8: Save CSV
import pandas as pd

def save_csv(rows, algo_name):
    csv_path = OUTPUT_DIR / f'results_{algo_name}.csv'
    df = pd.DataFrame(rows, columns=['cluster_ID', 'files', 'title', 'description'])
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f'Saved: {csv_path}')
    return df

df_arc = save_csv(cluster_results, CURRENT_ALGO)
df_arc

Saved: /content/drive/MyDrive/week4_outputs/results_ARC.csv


,cluster_ID,files,title,description
0,Cluster_0,ConfiguredNodeLabels.java; QueueNodeLabelsSett...,Hadoop YARN Capacity Scheduler - Node Label Ma...,The Hadoop YARN Capacity Scheduler architectur...
1,Cluster_1,AbstractAutoCreatedLeafQueue.java; AbstractMan...,Hadoop YARN Capacity Scheduler Architecture,The Hadoop YARN Capacity Scheduler implements ...
2,Cluster_2,AbstractContainerAllocator.java; AppPriorityAC...,Hadoop YARN Capacity Scheduler Architecture,The Hadoop YARN Capacity Scheduler is a sophis...
3,Cluster_3,AbsoluteResourceCapacityCalculator.java; Abstr...,Hadoop YARN Capacity Scheduler Architecture,The Hadoop YARN Capacity Scheduler is a sophis...
4,Cluster_4,CSConfigurationProvider.java; ConfigurationPro...,Hadoop YARN Capacity Scheduler Configuration M...,The Hadoop YARN Capacity Scheduler Configurati...
5,Cluster_5,AllocationState.java; CSAMContainerLaunchDiagn...,Hadoop YARN Capacity Scheduler Architecture,The Hadoop YARN Capacity Scheduler Architectur...
6,Cluster_6,ResourceVector.java,Hadoop YARN Capacity Scheduler Resource Manage...,The Hadoop YARN Capacity Scheduler architectur...
7,Cluster_7,GeneratePojos.java; LegacyMappingRuleToJson.ja...,Hadoop YARN Capacity Scheduler - Resource Mana...,The Hadoop YARN Capacity Scheduler's resource ...
8,Cluster_8,CSQueuePreemptionSettings.java,Hadoop YARN Capacity Scheduler Preemption Mana...,The Hadoop YARN Capacity Scheduler Preemption ...
9,Cluster_9,AbstractCSQueue.java; AbstractLeafQueue.java; ...,Hadoop YARN Capacity Scheduler Architecture,The Hadoop YARN Capacity Scheduler implements ...


In [ ]:
# Cell 9: Download CSVs from Drive
from google.colab import files

for f in OUTPUT_DIR.iterdir():
    if f.suffix == '.csv':
        print(f'Downloading {f.name}...')
        files.download(str(f))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 10: (Optional) Run ACDC — upload acdc_clusters.rsf to /content/ first
CURRENT_ALGO = 'ACDC'

if not RSF_FILES[CURRENT_ALGO].exists():
    print('acdc_clusters.rsf not found — skipping.')
else:
    a = parse_rsf(RSF_FILES[CURRENT_ALGO])
    c = build_cluster_to_files(a, SOURCE_ROOT)
    fs = run_leaf_summarisation(CURRENT_ALGO, c)
    ss = run_subdir_summarisation(CURRENT_ALGO, fs, c)
    r  = run_cluster_summarisation(CURRENT_ALGO, fs, ss, c)
    save_csv(r, CURRENT_ALGO)

  Format detected: ACDC
  Clusters: ['Cluster_0', 'Cluster_1', 'Cluster_2', 'Cluster_3', 'Cluster_4', 'Cluster_5', 'Cluster_6']
  Matched 279/749 files to clusters.
  [1/279] Cluster_0 -> DBManager.java


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  [2/279] Cluster_0 -> ResourceManager.java
  [3/279] Cluster_0 -> ConfigurationMutationACLPolicy.java
  [4/279] Cluster_0 -> ConfigurationMutationACLPolicyFactory.java
  [5/279] Cluster_0 -> ConfigurationUpdateAssembler.java
  [6/279] Cluster_0 -> FSSchedulerConfigurationStore.java
  [7/279] Cluster_0 -> FileBasedCSConfigurationProvider.java
  [8/279] Cluster_0 -> InMemoryConfigurationStore.java
  [9/279] Cluster_0 -> LeveldbConfigurationStore.java
  [10/279] Cluster_0 -> MutableCSConfigurationProvider.java
  [11/279] Cluster_0 -> QueueAdminConfigurationMutationACLPolicy.java
  [12/279] Cluster_0 -> QueueCapacityConfigParser.java
  [13/279] Cluster_0 -> YarnConfStoreVersionIncompatibleException.java
  [14/279] Cluster_0 -> YarnConfigurationStore.java
  [15/279] Cluster_0 -> YarnConfigurationStoreFactory.java
  [16/279] Cluster_0 -> ZKConfigurationStore.java
  [17/279] Cluster_0 -> RMWebServices.java
  [18/279] Cluster_1 -> MappingRule.java
  [19/279] Cluster_1 -> MappingRuleAction.jav

In [ ]:
# Cell 11: (Optional) Run LIMBO — upload limbo_clusters.rsf to /content/ first
CURRENT_ALGO = 'LIMBO'

if not RSF_FILES[CURRENT_ALGO].exists():
    print('limbo_clusters.rsf not found — skipping.')
else:
    a = parse_rsf(RSF_FILES[CURRENT_ALGO])
    c = build_cluster_to_files(a, SOURCE_ROOT)
    fs = run_leaf_summarisation(CURRENT_ALGO, c)
    ss = run_subdir_summarisation(CURRENT_ALGO, fs, c)
    r  = run_cluster_summarisation(CURRENT_ALGO, fs, ss, c)
    save_csv(r, CURRENT_ALGO)

  Format detected: LIMBO
  Clusters: ['Cluster_0', 'Cluster_1', 'Cluster_10', 'Cluster_11', 'Cluster_12', 'Cluster_13', 'Cluster_14', 'Cluster_15', 'Cluster_16', 'Cluster_17', 'Cluster_18', 'Cluster_19', 'Cluster_2', 'Cluster_3', 'Cluster_4', 'Cluster_5', 'Cluster_6', 'Cluster_7', 'Cluster_8', 'Cluster_9']
  Matched 279/749 files to clusters.
  [1/279] Cluster_0 -> AdminService.java
  [2/279] Cluster_0 -> DBManager.java
  [3/279] Cluster_0 -> RMContext.java
  [4/279] Cluster_0 -> RMCriticalThreadUncaughtExceptionHandler.java
  [5/279] Cluster_0 -> SchedulingEditPolicy.java
  [6/279] Cluster_0 -> SchedulingMonitorManager.java
  [7/279] Cluster_0 -> RMNodeLabelsManager.java
  [8/279] Cluster_0 -> ApplicationPlacementContext.java
  [9/279] Cluster_0 -> PlacementFactory.java
  [10/279] Cluster_0 -> PlacementManager.java
  [11/279] Cluster_0 -> PlacementRule.java
  [12/279] Cluster_0 -> MappingRule.java
  [13/279] Cluster_0 -> MappingRuleAction.java
  [14/279] Cluster_0 -> MappingRuleAction

In [ ]:
# Cell 12: Debug — run this if you see 0 files matched in Cell 5
print('=== RSF key sample (first 5) ===')
for k in list(assignments.keys())[:5]:
    print(' ', k)

print('\n=== Reconstructed file key sample (first 5) ===')
for fpath in list(SOURCE_ROOT.rglob('*.java'))[:5]:
    rel    = fpath.relative_to(SOURCE_ROOT)
    dotted = str(rel.with_suffix('')).replace('/', '.').replace('\\', '.')
    print(f'  {dotted}')

print('\nIf formats do not match, adjust SOURCE_ROOT in Cell 2.')

=== RSF key sample (first 5) ===
  org.apache.hadoop.yarn.server.resourcemanager.scheduler.capacity.AbsoluteResourceCapacityCalculator
  org.apache.hadoop.yarn.server.resourcemanager.scheduler.capacity.AbstractAutoCreatedLeafQueue
  org.apache.hadoop.yarn.server.resourcemanager.scheduler.capacity.AbstractCSQueue
  org.apache.hadoop.yarn.server.resourcemanager.scheduler.capacity.AbstractLeafQueue
  org.apache.hadoop.yarn.server.resourcemanager.scheduler.capacity.AbstractManagedParentQueue

=== Reconstructed file key sample (first 5) ===
  org.apache.hadoop.yarn.server.resourcemanager.GenericEventTypeMetricsManager
  org.apache.hadoop.yarn.server.resourcemanager.RMAppManagerEvent
  org.apache.hadoop.yarn.server.resourcemanager.RMAppManager
  org.apache.hadoop.yarn.server.resourcemanager.RMContextImpl
  org.apache.hadoop.yarn.server.resourcemanager.AdminService

If formats do not match, adjust SOURCE_ROOT in Cell 2.
